# Milestone 1 - crawl one member's range on Google Colab

Runs the crawler end to end on a **CPU-only** Colab runtime, the benchmark environment of the
project. See `docs/guideme.md` for the full walkthrough.

1. Use a **CPU** runtime (Runtime -> Change runtime type -> CPU).
2. Optional: add `S2_API_KEY` to Colab Secrets (key icon). Without it the crawl still works.
3. Set `MEMBER` below, then **Run all**. Keep the notebook open until the last cell finishes.

Every step is resumable. If the runtime disconnects, reconnect and **Run all** again: the crawl
restores its latest Google Drive backup and continues where it stopped.

In [ ]:
MEMBER = "m1"  # your range: m1, m2, m3 or m4
REPO_URL = "https://github.com/ttphat-fitus/IntroDS.git"
BRANCH = "main"

# Leave these as they are for the real run
USE_DRIVE = True  # back up to Google Drive (needed to survive a disconnect)
LIMIT = None  # e.g. 20 to try the pipeline on the first 20 IDs of each span

BACKUP_DIR = (
    f"/content/drive/MyDrive/introds-backup/{MEMBER}" if USE_DRIVE else f"/content/backup/{MEMBER}"
)
DATA_DIR = "/content/data"

## 1. Hardware

In [ ]:
import os
import platform
import shutil
import signal
import subprocess
import time

import psutil

print(platform.platform(), "| CPUs:", os.cpu_count())
print("RAM GB:", round(psutil.virtual_memory().total / 1e9, 1))
print("Disk free GB:", round(shutil.disk_usage("/content").free / 1e9, 1))
gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True, check=False)
print(gpu.stdout or "No GPU (expected: the benchmark is CPU-only)")

## 2. Code and dependencies

In [ ]:
if not os.path.exists("/content/introds"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "/content/introds"], check=True)
elif os.path.exists("/content/introds/.git"):
    subprocess.run(["git", "-C", "/content/introds", "pull", "--ff-only"], check=True)
subprocess.run("pip install -q -r /content/introds/src/requirements.txt", shell=True, check=True)
if not os.path.exists("/usr/bin/time"):  # GNU time: auditable wall time and peak RAM
    subprocess.run("apt-get -qq install -y time > /dev/null", shell=True, check=False)
print(
    "code:",
    subprocess.run(
        ["git", "-C", "/content/introds", "log", "-1", "--oneline"],
        capture_output=True,
        text=True,
        check=False,
    ).stdout.strip(),
)

## 3. Semantic Scholar API key (optional)

Read from Colab Secrets when available; otherwise you may paste it, or press Enter to skip.
References are fetched in batches of 100 papers, so the crawl works fine without a key.

In [ ]:
from getpass import getpass

if "S2_API_KEY" not in os.environ:
    try:
        from google.colab import userdata

        os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY") or ""
    except Exception:  # noqa: BLE001 - secret missing or not shared with this notebook
        os.environ["S2_API_KEY"] = getpass("S2_API_KEY (Enter to skip): ").strip()
os.environ["INTRODS_DATA_DIR"] = DATA_DIR
print(
    "Semantic Scholar key:", "set" if os.environ["S2_API_KEY"] else "not set (slower, still fine)"
)

## 4. Google Drive backup

Mounts Drive (a sign-in window appears the first time), then restores the latest backup if this
runtime is fresh. Backups are written to `MyDrive/introds-backup/<MEMBER>` every 20 minutes.

In [ ]:
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
os.makedirs(BACKUP_DIR, exist_ok=True)
if not os.path.exists(f"{DATA_DIR}/crawl.sqlite") and os.path.exists(f"{BACKUP_DIR}/crawl.sqlite"):
    subprocess.run(
        f"cd /content/introds/src && python -m introds restore {BACKUP_DIR}", shell=True, check=True
    )
    print("Restored the previous progress from", BACKUP_DIR)
else:
    print("Starting from", "existing local data" if os.path.exists(DATA_DIR) else "scratch")

## 5. Crawl

This cell runs for several hours and prints a progress line every 5 minutes. It must stay
running: Colab shuts down runtimes whose notebook is idle. To stop early, interrupt the cell
(the crawler finishes the current item and exits cleanly); run it again later to continue.

In [ ]:
timer = f"/usr/bin/time -v -o {BACKUP_DIR}/time_{MEMBER}_{int(time.time())}.txt "
limit = f" --limit {LIMIT}" if LIMIT else ""
cmd = (
    f"cd /content/introds/src && {timer if os.path.exists('/usr/bin/time') else ''}"
    f"python -m introds crawl --member {MEMBER} --backup-dir {BACKUP_DIR} --backup-every 20{limit}"
)
proc = subprocess.Popen(
    cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
try:
    with open(f"{BACKUP_DIR}/crawl_{MEMBER}.log", "a") as log:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
except KeyboardInterrupt:
    print("Stopping after the current item...")
    proc.send_signal(signal.SIGINT)
print("crawl exited with code", proc.wait())

## 6. Statistics for the report

Writes `stats/` next to the backup. When it is done, share the Drive folder
`introds-backup/<MEMBER>` with the team lead (see `docs/guideme.md`).

In [ ]:
subprocess.run(
    f"cd /content/introds/src && python -m introds stats --member {MEMBER} --out {BACKUP_DIR}/stats"
    f" > /dev/null && cp {BACKUP_DIR}/time_{MEMBER}_*.txt {BACKUP_DIR}/stats/ 2>/dev/null; true",
    shell=True,
    check=True,
)
with open(f"{BACKUP_DIR}/stats/summary.json") as f:
    print(f.read())